In [56]:
import mysql.connector
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\erfpPROD.ini')

In [57]:
host=config['erfpPROD']['host']
user=config['erfpPROD']['user']
pwd=config['erfpPROD']['pwd']
database=config['erfpPROD']['database']
print("SUCCESS")

In [58]:
conn = mysql.connector.connect(
          host=host,
          user=user,
          passwd=pwd,
          database=database)
cursor = conn.cursor()

In [90]:
sql_select_query = """SELECT 
      htl.HOTEL_ID_VALUE AS HOTEL_ID
    , htl.HOTEL_NAME_NAME AS HOTEL_NAME
    , rfp.ID_VALUE AS RFP_ID
    , rfp.NAME AS RFP_NAME
    , rfp.SEASON AS RFP_SEASON
    , tr.CURRENCY AS TARGET_RATE_CURRENCY
    , tr.RATE_TYPE
    , tr.BREAKFAST_TYPE
    , tr.AMOUNT AS TARGET_RATE
    , htl.MARKET_PLACE_HOTEL_STATUS
    , htl.AVERAGE_RATE_IN_EUR_AMOUNT
    , rfp.IS_TEST_RFP
    , rfp.STATUS
    ,trim(DATE(rfp.ENTITY_STATUS_CREATING_DATE)) AS RFP_CREATION_DATE
  FROM erfp_target_rate tr
LEFT JOIN erfp_hotel_for_rfpentity htl ON tr.OR_HOTEL_VALUE = htl.ID_VALUE
JOIN erfp_rfpentity rfp ON rfp.ID_VALUE = htl.OR_RFP_VALUE
WHERE rfp.SEASON IN (2019)
    AND htl.MARKET_PLACE_HOTEL_STATUS IN('INVITED', 'SIGNED_UP', 'SOLICITED' ) 
    AND IS_TEST_RFP IN (0)
ORDER BY HOTEL_ID"""

cursor.execute(sql_select_query)

QUERY = cursor.fetchall()
print('Total Row(s):', cursor.rowcount)

# Importing data into a DataFrame
import pandas as pd
erfp_df = pd.DataFrame()
a=[]
for row in QUERY:
        a.append(row)

erfp_df = pd.DataFrame(a)
df_col_names =  [i[0] for i in cursor.description]
erfp_df.columns = df_col_names

In [60]:
cursor.description[0][0]

In [74]:
erfp_df.head()

In [40]:
syb_df = pd.read_excel('C:/Users/USER/Documents/misc/target_rates_syb_prod.xlsx',
                       sheet_name='Grid Results', 
                       encoding='utf-8')

In [73]:
import pyexasol
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\ExasolPROD.ini')

dsn=config['exasolPROD']['dsn']
user=config['exasolPROD']['user']
pwd=config['exasolPROD']['pwd']
schema=config['exasolPROD']['schema']
print("SUCCESS")

# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
sql_query =  "select * from dwhbil.LKP_CURRENCY_EXCHANGE_RATE_HIST WHERE YEAR(CURRENCY_EXCHANGE_RATE_DATE) >= 2018"
QUERY = connect.execute(sql_query)

# Importing data into a DataFrame
import pandas as pd
exh_df = pd.DataFrame()
a=[]
for row in QUERY:
    a.append(row)
#print(len(a))
exh_df = pd.DataFrame(a)
df_col_names = QUERY.col_names
exh_df.columns = df_col_names
exh_df.head()

In [94]:
add_exch = pd.merge(left = erfp_df, 
                    right = exh_df, 
                    left_on = ['RFP_CREATION_DATE','TARGET_RATE_CURRENCY'], 
                    right_on = ['CURRENCY_EXCHANGE_RATE_DATE','CURRENCY_ISO'], 
                    how = 'inner')
add_exch.TARGET_RATE = add_exch.TARGET_RATE.astype
add_exch.CURRENCY_EXCHANGE_RATE = float(add_exch.CURRENCY_EXCHANGE_RATE)

In [96]:
add_exch['TARGET_RATE_EUR'] = add_exch.TARGET_RATE.astype('float64')/add_exch.CURRENCY_EXCHANGE_RATE.astype('float64')
add_exch.head(10)

In [97]:
result_df = pd.merge(left = add_exch, right = syb_df, on = ['RFP_ID', 'HOTEL_ID'], how = 'left')
result_df.head(100)

In [98]:
# Write in excel file
file = "C:\\Users\\USER\\Documents\\misc\\target_rates_normal.xlsx" 
result_df.to_excel(file, sheet_name='Target_rates', header=True, encoding='utf-8', index=False, freeze_panes=(1,0))
    